In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [8]:
# data: https://www.kaggle.com/datasets/quantbruce/real-estate-price-prediction
column_names = ['No', 'transaction_date', 'house_age', 'distance_to_mrt', 'num_convenience_stores', 'latitude', 'longitude', 'house_price']
data = pd.read_csv('../datasets/Real estate.csv', skiprows=1, names=column_names)
data.head()

,No,transaction_date,house_age,distance_to_mrt,num_convenience_stores,latitude,longitude,house_price
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


In [13]:
# data split
X = data.drop(['house_price', 'No'], axis=1)
y = data['house_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# hyperparameter tuning (search space)
param_dist = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [5, 10, 20, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None]
}

# cv
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# model based on best estimators
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# finding best params (hyperparameter tuning) and train final model/fit
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: Pipeline(steps=[('scaler', StandardScaler()),
                ('rf',
                 RandomForestRegressor(max_depth=20, max_features='log2',
                                       min_samples_leaf=4, min_samples_split=10,
                                       n_estimators=50, n_jobs=-1,
                                       random_state=42))])
Best CV score (on training): 0.6746


In [18]:
# performance metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

print(f"\n📊 Cross Validation:")
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 73.5% of the variation in house price of unit area
   • Only 26.5% of variation is unexplained (due to other factors)
Training R²: 0.7962
Test R²: 0.7354
⚠️  Warning: Difference of 0.061 between training and testing R²


📊 Cross Validation:
Cross-validation R²: 0.727 (+/- 0.103)


📊 MSE:
Training MSE: 39.04
Test MSE: 44.24


📊 RMSE
   • Predictions are off by ±6.65 on average
   • In other words, 68% of predictions fall within 6.65 of actual value
   • 95% of predictions fall within 13.30 of actual value
Training RMSE: 6.25
Testing RMSE: 6.65


📊 MAE:
Training MAE: 3.79
Testing MAE: 4.62


In [20]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.named_steps['rf'].feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance)



FEATURE IMPORTANCE
                  Feature  Importance
2         distance_to_mrt    0.309536
4                latitude    0.236205
3  num_convenience_stores    0.159846
5               longitude    0.148449
1               house_age    0.114346
0        transaction_date    0.031618


#### I ran linear regression on this dataset here is the comparison:

| Metric | Linear Regression | Random Forest | Winner |
|--------|-------------------|---------------|--------|
| **Test R²** | 0.6811 (68.1%) | 0.7354 (73.5%) | ✅ RF (+5.4%) |
| **Test RMSE** | 7.31 | 6.65 | ✅ RF (-0.66) |
| **Test MAE** | 5.31 | 4.62 | ✅ RF (-0.69) |
| **Test MSE** | 53.51 | 44.24 | ✅ RF (-9.27) |